# 04 - Entrenamiento de Modelos

Este notebook entrena y evalúa múltiples modelos de Machine Learning para la detección de phishing/smishing.

## Contenido
1. Preparación de datos
2. Modelos clásicos (Naive Bayes, SVM, Random Forest, XGBoost)
3. Evaluación y comparación
4. Guardado de modelos

In [ ]:
# Imports
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

from src.data import SMSLoader, TextPreprocessor
from src.features import TextFeatureExtractor
from src.models import ClassicalModels
from src.evaluation import ModelEvaluator
from src.utils.visualization import Visualizer

## 1. Preparación de Datos

In [ ]:
# Cargar datos
sms_loader = SMSLoader(data_dir='../data')

try:
    df = sms_loader.load_processed('sms_preprocessed.csv')
    text_col = 'text_clean_ml'
except FileNotFoundError:
    try:
        df = sms_loader.load_uci_sms_spam()
    except:
        from src.data.sms_loader import create_sample_dataset
        df = create_sample_dataset()
    
    # Preprocesar
    preprocessor = TextPreprocessor(lowercase=True, replace_urls=True)
    df['text_clean_ml'] = preprocessor.preprocess_batch(df['body'])
    text_col = 'text_clean_ml'

print(f"Dataset cargado: {len(df)} muestras")
print(f"Distribución de clases: {df['label'].value_counts().to_dict()}")

In [ ]:
# Extraer características TF-IDF
feature_extractor = TextFeatureExtractor(
    method='tfidf',
    max_features=5000,
    ngram_range=(1, 2),
    use_char_ngrams=True
)

X = feature_extractor.fit_transform(df[text_col])
y = df['label'].values

print(f"Shape de características: {X.shape}")

In [ ]:
# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")

## 2. Entrenamiento de Modelos Clásicos

In [ ]:
# Inicializar gestor de modelos
models = ClassicalModels(model_dir='../models')

# Lista de modelos a entrenar
model_names = ['naive_bayes', 'logistic_regression', 'linear_svm', 'random_forest']

# Intentar añadir XGBoost si está disponible
try:
    import xgboost
    model_names.append('xgboost')
except ImportError:
    print("XGBoost no disponible")

In [ ]:
# Entrenar todos los modelos
print("Entrenando modelos...\n")

for model_name in model_names:
    print(f"Entrenando {model_name}...")
    models.train(model_name, X_train, y_train)
    print(f"  ✓ {model_name} entrenado")

print("\nEntrenamiento completado!")

## 3. Evaluación de Modelos

In [ ]:
# Inicializar evaluador
evaluator = ModelEvaluator(output_dir='../reports/results')

# Evaluar cada modelo
results = {}

for model_name in model_names:
    print(f"\nEvaluando {model_name}...")
    
    # Predicciones
    y_pred = models.predict(model_name, X_test)
    y_proba = models.predict_proba(model_name, X_test)
    
    # Métricas
    metrics = evaluator.evaluate(
        y_true=y_test,
        y_pred=y_pred,
        y_proba=y_proba,
        model_name=model_name
    )
    
    results[model_name] = metrics
    evaluator.print_summary(model_name)

In [ ]:
# Comparación de modelos
comparison_df = evaluator.compare_models()
print("\nComparación de Modelos:")
display(comparison_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(4))

## 4. Visualización de Resultados

In [ ]:
# Visualizador
viz = Visualizer(output_dir='../reports/figures')

# Gráfico de comparación
viz.plot_metrics_comparison(
    comparison_df,
    metrics=['accuracy', 'precision', 'recall', 'f1'],
    title='Comparación de Modelos',
    save_path='model_comparison.png'
)

In [ ]:
# Mejor modelo
best_model_name, best_score = evaluator.get_best_model(metric='f1')
print(f"\nMejor modelo: {best_model_name} (F1: {best_score:.4f})")

# Matriz de confusión del mejor modelo
best_result = evaluator.results[best_model_name]
viz.plot_confusion_matrix(
    best_result['confusion_matrix'],
    title=f'Matriz de Confusión - {best_model_name}',
    save_path=f'confusion_matrix_{best_model_name}.png'
)

## 5. Guardado de Modelos

In [ ]:
# Guardar todos los modelos
models.save_all_models()
print("Modelos guardados exitosamente!")

# Guardar el extractor de características
feature_extractor.save('../models/tfidf_vectorizer.pkl')
print("Vectorizador TF-IDF guardado!")

In [ ]:
# Guardar resultados
evaluator.save_comparison('model_comparison.csv')
print("Resultados guardados!")

## 6. Resumen

### Resultados del entrenamiento:

| Modelo | Accuracy | Precision | Recall | F1 |
|--------|----------|-----------|--------|----|
| (Completar con resultados reales) |

### Conclusiones:
- El mejor modelo es: [modelo]
- F1-Score alcanzado: [score]

### Próximos pasos:
- Comparación completa (notebook 05)
- Visualización de resultados (notebook 06)
- Entrenamiento de modelos de Deep Learning